# cryoDRGN Latent Space Comparison (Two Independent .pkl Files)

Compares latent space embeddings from **two separate** cryoDRGN VAE latent variable files
(e.g. two different runs, or two pre-split subsets that were each saved to their own `.pkl`).

Unlike `cryoDRGN_LatentSPACE_Analysis.ipynb` (which splits a single combined `z.pkl` by index
into OA/OEA), this notebook loads `PATH_A` and `PATH_B` independently, so the two datasets do
not need to be the same size or come from the same run.

Set the configuration in the cell below, then run the notebook top to bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from scipy import stats
from cryodrgn import utils

# --- User configuration ---
PATH_A = Path("Z-values/z_A.pkl")
PATH_B = Path("Z-values/z_B.pkl")
LABEL_A = "Dataset A"
LABEL_B = "Dataset B"
OUTPUT_DIR = Path("comparison_output")
OUTPUT_DIR.mkdir(exist_ok=True)

dataA = utils.load_pkl(PATH_A)
dataB = utils.load_pkl(PATH_B)

print(f"{LABEL_A}: {dataA.shape} loaded from {PATH_A}")
print(f"{LABEL_B}: {dataB.shape} loaded from {PATH_B}")


## Vector magnitude (L2 norm) comparison

In [ ]:
magnitudes_A = np.linalg.norm(dataA, ord=2, axis=1)
magnitudes_B = np.linalg.norm(dataB, ord=2, axis=1)

plt.figure(figsize=(10, 6))
plt.hist(magnitudes_A, bins=100, alpha=0.5, label=LABEL_A)
plt.hist(magnitudes_B, bins=100, alpha=0.5, label=LABEL_B)
plt.title('Histogram of Vector Magnitudes (L2 Norm)')
plt.xlabel('Magnitude')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "magnitude_histogram.png")
plt.show()


## Pairwise dimension plots

In [ ]:
dfA = pd.DataFrame(dataA, columns=[f'dim{i+1}' for i in range(dataA.shape[1])])
dfA['dataset'] = LABEL_A
dfB = pd.DataFrame(dataB, columns=[f'dim{i+1}' for i in range(dataB.shape[1])])
dfB['dataset'] = LABEL_B
df_combined = pd.concat([dfA, dfB], ignore_index=True)

sns.set_theme(style="whitegrid")
sns.pairplot(df_combined, hue='dataset', plot_kws={'alpha': 0.4, 's': 10})
plt.savefig(OUTPUT_DIR / "pairplot.png")
plt.show()


## Random subsets & magnitude statistics

Two random subsets are drawn from each dataset (for reproducibility checks later). The magnitudes computed above are compared statistically and shown as a violin plot.

In [ ]:
# Use a common subset size so A/B subsets are directly comparable even if the two
# datasets have different total sizes.
SUBSET_SIZE = min(30000, len(dataA), len(dataB))

def random_subsets(data, size, seed_1=42, seed_2=43):
    rng_1 = np.random.default_rng(seed_1)
    rng_2 = np.random.default_rng(seed_2)
    idx_1 = rng_1.choice(len(data), size=size, replace=False)
    idx_2 = rng_2.choice(len(data), size=size, replace=False)
    return data[idx_1], data[idx_2]

A_subset_1, A_subset_2 = random_subsets(dataA, SUBSET_SIZE)
B_subset_1, B_subset_2 = random_subsets(dataB, SUBSET_SIZE)

plt.figure(figsize=(10, 6))
plt.violinplot([magnitudes_A, magnitudes_B], showmeans=True)
plt.title('Violin Plot of Vector Magnitudes')
plt.xticks([1, 2], [LABEL_A, LABEL_B])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "violin_plot.png")
plt.show()

t_stat, p_value = stats.ttest_ind(magnitudes_A, magnitudes_B)
print('T-test')
print('T-Statistic:', t_stat)
print('P-Value:', p_value)

u_stat, p_value = stats.mannwhitneyu(magnitudes_A, magnitudes_B, alternative='two-sided')
print('Mann-Whitney U-Test')
print('U-Statistic:', u_stat)
print('P-Value:', p_value)

f_stat, p_value = stats.f_oneway(magnitudes_A, magnitudes_B)
print('ANOVA Test')
print('F-Statistic:', f_stat)
print('P-Value:', p_value)

print(f'Mean {LABEL_A} Magnitude:', np.mean(magnitudes_A), 'SD', np.std(magnitudes_A))
print(f'Mean {LABEL_B} Magnitude:', np.mean(magnitudes_B), 'SD', np.std(magnitudes_B))


## UMAP embedding

Embeddings are cached to `OUTPUT_DIR` so re-running the notebook doesn't recompute them.

In [ ]:
import umap
import joblib

def perform_and_save_umap(data, filename, **umap_kwargs):
    filename = Path(filename)
    if filename.exists():
        print(f"Loading UMAP embeddings from {filename}...")
        return joblib.load(filename)
    print(f"Performing UMAP and saving results to {filename}...")
    reducer = umap.UMAP(n_components=2, metric='euclidean', min_dist=0.1, n_neighbors=50,
                         spread=1.5, learning_rate=0.5, negative_sample_rate=10,
                         init='pca', random_state=42, **umap_kwargs)
    embedding = reducer.fit_transform(data)
    joblib.dump(embedding, filename)
    return embedding

umap_A = perform_and_save_umap(dataA, OUTPUT_DIR / "umap_A.pkl")
umap_B = perform_and_save_umap(dataB, OUTPUT_DIR / "umap_B.pkl")
umap_A_subset_1 = perform_and_save_umap(A_subset_1, OUTPUT_DIR / "umap_A_subset_1.pkl")
umap_A_subset_2 = perform_and_save_umap(A_subset_2, OUTPUT_DIR / "umap_A_subset_2.pkl")
umap_B_subset_1 = perform_and_save_umap(B_subset_1, OUTPUT_DIR / "umap_B_subset_1.pkl")
umap_B_subset_2 = perform_and_save_umap(B_subset_2, OUTPUT_DIR / "umap_B_subset_2.pkl")


## t-SNE embedding

In [ ]:
from sklearn.manifold import TSNE

def perform_and_save_tsne(data, filename, perplexity=100, n_iter=1000):
    filename = Path(filename)
    if filename.exists():
        print(f"Loading t-SNE embeddings from {filename}...")
        return joblib.load(filename)
    print(f"Performing t-SNE and saving results to {filename}...")
    tsne = TSNE(perplexity=perplexity, n_iter=n_iter, random_state=42)
    embedding = tsne.fit_transform(data)
    joblib.dump(embedding, filename)
    return embedding

tsne_A = perform_and_save_tsne(dataA, OUTPUT_DIR / "tsne_A.pkl")
tsne_B = perform_and_save_tsne(dataB, OUTPUT_DIR / "tsne_B.pkl")
tsne_A_subset_1 = perform_and_save_tsne(A_subset_1, OUTPUT_DIR / "tsne_A_subset_1.pkl")
tsne_A_subset_2 = perform_and_save_tsne(A_subset_2, OUTPUT_DIR / "tsne_A_subset_2.pkl")
tsne_B_subset_1 = perform_and_save_tsne(B_subset_1, OUTPUT_DIR / "tsne_B_subset_1.pkl")
tsne_B_subset_2 = perform_and_save_tsne(B_subset_2, OUTPUT_DIR / "tsne_B_subset_2.pkl")


## Reproducibility & cross-dataset similarity

Compares pairwise-distance matrices (via Pearson correlation) between: two random subsets of
the same dataset (a reproducibility check), and between the two datasets themselves.

In [ ]:
from scipy.stats import pearsonr
from sklearn.metrics.pairwise import euclidean_distances

def distance_similarity(emb_1, emb_2, name):
    dist_1 = euclidean_distances(emb_1)
    dist_2 = euclidean_distances(emb_2)
    score, _ = pearsonr(dist_1.flatten(), dist_2.flatten())
    print(f'Similarity score for {name}: {score}')
    return score

distance_similarity(tsne_A_subset_1, tsne_A_subset_2, f'{LABEL_A} random subsets')
distance_similarity(tsne_B_subset_1, tsne_B_subset_2, f'{LABEL_B} random subsets')

if len(dataA) == len(dataB):
    distance_similarity(tsne_A, tsne_B, f'{LABEL_A} vs {LABEL_B} (full)')
else:
    print(f'Skipping full-dataset {LABEL_A} vs {LABEL_B} comparison (different sizes); using subsets instead.')
distance_similarity(tsne_A_subset_1, tsne_B_subset_1, f'{LABEL_A} vs {LABEL_B} (subset)')


## UMAP / t-SNE subset reproducibility plots

In [ ]:
plt.figure(figsize=(20, 10))

plt.subplot(2, 2, 1)
plt.scatter(umap_A_subset_1[:, 0], umap_A_subset_1[:, 1], alpha=0.5, label=f'{LABEL_A} Subset 1')
plt.scatter(umap_A_subset_2[:, 0], umap_A_subset_2[:, 1], alpha=0.5, label=f'{LABEL_A} Subset 2')
plt.title(f'UMAP of {LABEL_A} Subsets')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.legend()

plt.subplot(2, 2, 2)
plt.scatter(umap_B_subset_1[:, 0], umap_B_subset_1[:, 1], alpha=0.5, label=f'{LABEL_B} Subset 1')
plt.scatter(umap_B_subset_2[:, 0], umap_B_subset_2[:, 1], alpha=0.5, label=f'{LABEL_B} Subset 2')
plt.title(f'UMAP of {LABEL_B} Subsets')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.legend()

plt.subplot(2, 2, 3)
plt.scatter(tsne_A_subset_1[:, 0], tsne_A_subset_1[:, 1], alpha=0.5, label=f'{LABEL_A} Subset 1')
plt.scatter(tsne_A_subset_2[:, 0], tsne_A_subset_2[:, 1], alpha=0.5, label=f'{LABEL_A} Subset 2')
plt.title(f'tSNE of {LABEL_A} Subsets')
plt.xlabel('tSNE 1')
plt.ylabel('tSNE 2')
plt.legend()

plt.subplot(2, 2, 4)
plt.scatter(tsne_B_subset_1[:, 0], tsne_B_subset_1[:, 1], alpha=0.5, label=f'{LABEL_B} Subset 1')
plt.scatter(tsne_B_subset_2[:, 0], tsne_B_subset_2[:, 1], alpha=0.5, label=f'{LABEL_B} Subset 2')
plt.title(f'tSNE of {LABEL_B} Subsets')
plt.xlabel('tSNE 1')
plt.ylabel('tSNE 2')
plt.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "umap_tsne_subsets.png")
plt.show()


## PCA

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=3)
data_pca_A = pca.fit_transform(dataA)
data_pca_B = pca.fit_transform(dataB)


## UMAP full-dataset plots

In [ ]:
umap_magnitudes_A = np.linalg.norm(umap_A, ord=2, axis=1)
umap_magnitudes_B = np.linalg.norm(umap_B, ord=2, axis=1)

plt.figure(figsize=(20, 30))

plt.subplot(3, 2, 1)
plt.title(f'UMAP of latent space {LABEL_A}')
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')
plt.scatter(umap_A[:, 0], umap_A[:, 1], alpha=0.9, label=LABEL_A, marker=".")

plt.subplot(3, 2, 2)
plt.hexbin(umap_A[:, 0], umap_A[:, 1], gridsize=50, cmap='Oranges', mincnt=1)
cbar = plt.colorbar()
cbar.set_label('Counts')
plt.title(f'UMAP Hexbin Visualization of {LABEL_A}')
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')

plt.subplot(3, 2, 3)
plt.title(f'UMAP of latent space {LABEL_B}')
plt.scatter(umap_B[:, 0], umap_B[:, 1], alpha=0.9, label=LABEL_B, marker=".")
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')

plt.subplot(3, 2, 4)
plt.hexbin(umap_B[:, 0], umap_B[:, 1], gridsize=50, cmap='Oranges', mincnt=1)
cbar = plt.colorbar()
cbar.set_label('Counts')
plt.title(f'UMAP Hexbin Visualization of {LABEL_B}')
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')

plt.subplot(3, 2, 5)
plt.scatter(umap_A[:, 0], umap_A[:, 1], alpha=0.9, label=LABEL_A, marker=".")
plt.scatter(umap_B[:, 0], umap_B[:, 1], alpha=0.1, label=LABEL_B, marker=".")
plt.title(f'UMAP of {LABEL_A} and {LABEL_B}')
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')
plt.legend()

plt.subplot(3, 2, 6)
plt.hist(umap_magnitudes_A, bins=100, alpha=0.5, label=LABEL_A)
plt.hist(umap_magnitudes_B, bins=100, alpha=0.5, label=LABEL_B)
plt.title('Histogram of UMAP Vector Magnitudes')
plt.xlabel('Magnitude')
plt.ylabel('Frequency')
plt.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "umap_visualizations.png")
plt.show()


## PCA plots

In [ ]:
plt.figure(figsize=(20, 20))

plt.subplot(2, 2, 1)
plt.scatter(data_pca_A[:, 0], data_pca_A[:, 1], alpha=0.9, label=LABEL_A, marker=".")
plt.scatter(data_pca_B[:, 0], data_pca_B[:, 1], alpha=0.3, label=LABEL_B, marker=".")
plt.legend()
plt.title('PCA of 1st and 2nd Principal Component')
plt.xlabel('Principal Component 0')
plt.ylabel('Principal Component 1')

plt.subplot(2, 2, 2)
plt.scatter(data_pca_A[:, 0], data_pca_A[:, 2], alpha=0.9, label=LABEL_A, marker=".")
plt.scatter(data_pca_B[:, 0], data_pca_B[:, 2], alpha=0.3, label=LABEL_B, marker=".")
plt.legend()
plt.title('PCA of 1st and 3rd Principal Component')
plt.xlabel('Principal Component 0')
plt.ylabel('Principal Component 2')

plt.subplot(2, 2, 3)
plt.scatter(data_pca_A[:, 1], data_pca_A[:, 2], alpha=0.9, label=LABEL_A, marker=".")
plt.scatter(data_pca_B[:, 1], data_pca_B[:, 2], alpha=0.3, label=LABEL_B, marker=".")
plt.legend()
plt.title('PCA of 2nd and 3rd Principal Component')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pca_visualizations.png")
plt.show()


## t-SNE full-dataset plots

In [ ]:
plt.figure(figsize=(20, 30))

plt.subplot(3, 2, 1)
plt.scatter(tsne_A[:, 0], tsne_A[:, 1], alpha=0.3, label=LABEL_A, marker=".")
plt.legend()
plt.title(f't-SNE Visualization of {LABEL_A}')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')

plt.subplot(3, 2, 2)
plt.hexbin(tsne_A[:, 0], tsne_A[:, 1], gridsize=40, cmap='Oranges', mincnt=1)
cbar = plt.colorbar()
cbar.set_label('Counts')
plt.title(f'tSNE Hexbin Visualization of {LABEL_A}')
plt.xlabel('tSNE Component 1')
plt.ylabel('tSNE Component 2')

plt.subplot(3, 2, 3)
plt.scatter(tsne_B[:, 0], tsne_B[:, 1], alpha=0.3, label=LABEL_B, marker=".")
plt.legend()
plt.title(f't-SNE Visualization of {LABEL_B}')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')

plt.subplot(3, 2, 4)
plt.hexbin(tsne_B[:, 0], tsne_B[:, 1], gridsize=40, cmap='Oranges', mincnt=1)
cbar = plt.colorbar()
cbar.set_label('Counts')
plt.title(f'tSNE Hexbin Visualization of {LABEL_B}')
plt.xlabel('tSNE Component 1')
plt.ylabel('tSNE Component 2')

plt.subplot(3, 2, 5)
plt.scatter(tsne_A[:, 0], tsne_A[:, 1], alpha=0.3, label=LABEL_A, marker=".")
plt.scatter(tsne_B[:, 0], tsne_B[:, 1], alpha=0.3, label=LABEL_B, marker=".")
plt.legend()
plt.title(f't-SNE Visualization of {LABEL_A} and {LABEL_B}')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "tsne_visualizations.png")
plt.show()


## Cosine similarity between embeddings

An additional comparison: cosine similarity between matched points in the t-SNE embeddings,
for subset-vs-subset reproducibility and dataset-vs-dataset comparison.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim_A_subsets = cosine_similarity(tsne_A_subset_1, tsne_A_subset_2)
cosine_sim_B_subsets = cosine_similarity(tsne_B_subset_1, tsne_B_subset_2)
cosine_sim_A_vs_B = cosine_similarity(tsne_A_subset_1, tsne_B_subset_1)

plt.figure(figsize=(20, 20))

plt.subplot(2, 2, 1)
sc1 = plt.scatter(tsne_A_subset_1[:, 0], tsne_A_subset_1[:, 1],
                   c=np.diag(cosine_sim_A_subsets), cmap='Oranges', s=10)
plt.colorbar(sc1, label='Cosine Similarity')
plt.title(f'tSNE Plot Colored by Cosine Similarity: {LABEL_A} subsets')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')

plt.subplot(2, 2, 2)
sc2 = plt.scatter(tsne_B_subset_1[:, 0], tsne_B_subset_1[:, 1],
                   c=np.diag(cosine_sim_B_subsets), cmap='Blues', s=10)
plt.colorbar(sc2, label='Cosine Similarity')
plt.title(f'tSNE Plot Colored by Cosine Similarity: {LABEL_B} subsets')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')

plt.subplot(2, 2, 3)
sc3 = plt.scatter(tsne_A_subset_1[:, 0], tsne_A_subset_1[:, 1],
                   c=np.diag(cosine_sim_A_vs_B), cmap='Greens', s=10)
plt.colorbar(sc3, label='Cosine Similarity')
plt.title(f'tSNE Plot Colored by Cosine Similarity: {LABEL_A} vs {LABEL_B}')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "tsne_cosine_similarity.png")
plt.show()

for name, matrix in [
    (f'{LABEL_A} random subsets', cosine_sim_A_subsets),
    (f'{LABEL_B} random subsets', cosine_sim_B_subsets),
    (f'{LABEL_A} vs {LABEL_B}', cosine_sim_A_vs_B),
]:
    print(f'Mean Cosine Similarity for {name}: {np.mean(matrix)}, SD: {np.std(matrix)}')
    positive = matrix[matrix > 0]
    print(f'  (positive values only): {np.mean(positive)}, SD: {np.std(positive)}')
